# Gut Microbiome Diversity — Validation Report
#
Validation of the gut_microbiome_diversity curated phenotype.
#
**Generated:** 2026-02-23
**Skill:** create-curated-phenotype v0.2.0
**Skills used:** hpp-datasets v1.1, databricks (MCP tool)
#
## Validation Gates
#
| Gate | Status |
|------|--------|
| 1. Correctness (comparison with recomputed sample) | TBD |
| 2. QC Gates (missingness, ranges, prevalence, duplicates, population coverage) | TBD |
| 3. Citation Verification | TBD |
| 4. Reproducibility | TBD |

In [1]:
import os
import re
import pandas as pd
import numpy as np
from databricks import sql
from dotenv import load_dotenv
from datetime import datetime, timezone

load_dotenv()

True

In [2]:
# Connect to Databricks
connection = sql.connect(
    server_hostname=os.getenv("DATABRICKS_HOST").replace("https://", "").rstrip("/"),
    http_path=f"/sql/1.0/warehouses/{os.getenv('DATABRICKS_SQL_WAREHOUSE_ID')}",
    access_token=os.getenv("DATABRICKS_TOKEN"),
)

[WARN] pyarrow is not installed by default since databricks-sql-connector 4.0.0,any arrow specific api (e.g. fetchmany_arrow) and cloud fetch will be disabled.If you need these features, please run pip install pyarrow or pip install databricks-sql-connector[pyarrow] to install


In [3]:
# Load output table
query = "SELECT * FROM ds.polaris.gut_microbiome_diversity"
cursor = connection.cursor()
cursor.execute(query)
output_df = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])
cursor.close()

print(f"Loaded output table: {len(output_df):,} participants")
print(f"Columns: {list(output_df.columns)}")

Loaded output table: 12,302 participants
Columns: ['participant_uuid', 'gut_microbiome_diversity__shannon_index', 'gut_microbiome_diversity__observed_species', 'gut_microbiome_diversity__diversity_category', 'gut_microbiome_diversity__quality_flag']


## Gate 1: Correctness
#
**Test:** Recompute diversity for a sample of participants and compare with output table

In [4]:
# Sample 100 random participants for validation
np.random.seed(42)
sample_participants = np.random.choice(output_df['participant_uuid'].values, size=min(100, len(output_df)), replace=False)

print(f"Selected {len(sample_participants)} participants for correctness validation")

Selected 100 participants for correctness validation


In [5]:
# Fetch raw species data for sampled participants
placeholders = ','.join([f"'{p}'" for p in sample_participants])
query = f"""
WITH most_recent_samples AS (
    SELECT
        participant_uuid,
        MAX(sample_id) as max_sample_id
    FROM ds.omics.gut_mb_metaphlan
    WHERE level = 'species'
        AND mock = false
        AND participant_uuid IN ({placeholders})
    GROUP BY participant_uuid
)
SELECT
    m.participant_uuid,
    m.species,
    m.relative_abundance
FROM ds.omics.gut_mb_metaphlan m
INNER JOIN most_recent_samples mrs
    ON m.participant_uuid = mrs.participant_uuid
    AND m.sample_id = mrs.max_sample_id
WHERE m.level = 'species'
    AND m.mock = false
    AND m.species IS NOT NULL
    AND m.relative_abundance > 0
ORDER BY m.participant_uuid, m.species
"""

cursor = connection.cursor()
cursor.execute(query)
raw_data = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])
cursor.close()

print(f"Fetched {len(raw_data):,} species records for validation sample")

Fetched 20,185 species records for validation sample


In [6]:
# Recompute Shannon index and observed species
def calculate_shannon(abundances):
    """Calculate Shannon index from relative abundances."""
    abundances = np.array(abundances)
    abundances = abundances[abundances > 0]
    if len(abundances) == 0:
        return np.nan
    proportions = abundances / abundances.sum()
    return -np.sum(proportions * np.log(proportions))

recomputed = []
for participant_uuid, group in raw_data.groupby('participant_uuid'):
    abundances = group['relative_abundance'].values
    shannon_index = calculate_shannon(abundances)
    observed_species = len(abundances)

    recomputed.append({
        'participant_uuid': participant_uuid,
        'shannon_recomputed': shannon_index,
        'species_recomputed': observed_species,
    })

recomputed_df = pd.DataFrame(recomputed)
print(f"Recomputed diversity for {len(recomputed_df)} participants")

Recomputed diversity for 100 participants


In [7]:
# Compare recomputed values with output table
comparison = output_df[output_df['participant_uuid'].isin(sample_participants)].copy()
comparison = comparison.merge(recomputed_df, on='participant_uuid', how='inner')

# Calculate differences
comparison['shannon_diff'] = np.abs(
    comparison['gut_microbiome_diversity__shannon_index'] - comparison['shannon_recomputed']
)
comparison['species_diff'] = np.abs(
    comparison['gut_microbiome_diversity__observed_species'] - comparison['species_recomputed']
)

print("\n" + "=" * 80)
print("CORRECTNESS VALIDATION RESULTS")
print("=" * 80)
print(f"\nSample size: {len(comparison)} participants")
print(f"\nShannon index differences:")
print(f"  Max difference: {comparison['shannon_diff'].max():.2e}")
print(f"  Mean difference: {comparison['shannon_diff'].mean():.2e}")
print(f"  Participants with diff > 1e-6: {(comparison['shannon_diff'] > 1e-6).sum()}")

print(f"\nObserved species differences:")
print(f"  Max difference: {comparison['species_diff'].max():.0f}")
print(f"  Mean difference: {comparison['species_diff'].mean():.2f}")
print(f"  Participants with diff > 0: {(comparison['species_diff'] > 0).sum()}")

# Gate 1 result
gate1_pass = (comparison['shannon_diff'].max() < 1e-6) and (comparison['species_diff'].max() == 0)
print(f"\n✓ Gate 1 (Correctness): {'PASS' if gate1_pass else 'FAIL'}")


CORRECTNESS VALIDATION RESULTS

Sample size: 100 participants

Shannon index differences:
  Max difference: 8.88e-15
  Mean difference: 1.91e-15
  Participants with diff > 1e-6: 0

Observed species differences:
  Max difference: 0
  Mean difference: 0.00
  Participants with diff > 0: 0

✓ Gate 1 (Correctness): PASS


## Gate 2: QC Gates
#
**Tests:**
1. Missingness within bounds (<5%)
2. Plausible ranges (Shannon 0-10, species 0-1000)
3. Prevalence aligns with epidemiology
4. No duplicate rows
5. All participant_uuid in population table

In [8]:
print("\n" + "=" * 80)
print("QC GATE VALIDATION")
print("=" * 80)

# Test 2.1: Missingness
print("\n2.1 Missingness Analysis:")
gate2_1_pass = True
for col in output_df.columns:
    missing_pct = output_df[col].isna().sum() / len(output_df) * 100
    status = "✓" if missing_pct < 5 else "✗"
    print(f"  {status} {col}: {missing_pct:.2f}% missing")
    if missing_pct >= 5:
        gate2_1_pass = False

print(f"\n✓ Gate 2.1 (Missingness <5%): {'PASS' if gate2_1_pass else 'FAIL'}")


QC GATE VALIDATION

2.1 Missingness Analysis:
  ✓ participant_uuid: 0.00% missing
  ✓ gut_microbiome_diversity__shannon_index: 0.00% missing
  ✓ gut_microbiome_diversity__observed_species: 0.00% missing
  ✓ gut_microbiome_diversity__diversity_category: 0.00% missing
  ✓ gut_microbiome_diversity__quality_flag: 0.00% missing

✓ Gate 2.1 (Missingness <5%): PASS


In [9]:
# Test 2.2: Plausible ranges
print("\n2.2 Plausible Ranges:")
shannon_col = 'gut_microbiome_diversity__shannon_index'
species_col = 'gut_microbiome_diversity__observed_species'

shannon_min, shannon_max = output_df[shannon_col].min(), output_df[shannon_col].max()
species_min, species_max = output_df[species_col].min(), output_df[species_col].max()

print(f"  Shannon index range: [{shannon_min:.2f}, {shannon_max:.2f}]")
print(f"  Observed species range: [{species_min:.0f}, {species_max:.0f}]")

gate2_2_pass = (0 <= shannon_min <= shannon_max <= 10) and (0 <= species_min <= species_max <= 1000)
status = "✓" if gate2_2_pass else "✗"
print(f"\n{status} Gate 2.2 (Plausible ranges): {'PASS' if gate2_2_pass else 'FAIL'}")


2.2 Plausible Ranges:
  Shannon index range: [0.62, 4.97]
  Observed species range: [5, 476]

✓ Gate 2.2 (Plausible ranges): PASS


In [10]:
# Test 2.3: Prevalence aligns with epidemiology
print("\n2.3 Prevalence Check:")
div_cat_col = 'gut_microbiome_diversity__diversity_category'
low_div_pct = (output_df[div_cat_col] == 'Low diversity').sum() / len(output_df) * 100
print(f"  Low diversity prevalence: {low_div_pct:.1f}%")
print(f"  Expected range: 8-20% (allowing variation from 10-15% baseline)")

gate2_3_pass = 8 <= low_div_pct <= 20
status = "✓" if gate2_3_pass else "✗"
print(f"\n{status} Gate 2.3 (Prevalence alignment): {'PASS' if gate2_3_pass else 'FAIL'}")


2.3 Prevalence Check:
  Low diversity prevalence: 12.3%
  Expected range: 8-20% (allowing variation from 10-15% baseline)

✓ Gate 2.3 (Prevalence alignment): PASS


In [11]:
# Test 2.4: No duplicate rows
print("\n2.4 Duplicate Check:")
n_duplicates = output_df['participant_uuid'].duplicated().sum()
print(f"  Duplicate participant_uuid values: {n_duplicates}")

gate2_4_pass = n_duplicates == 0
status = "✓" if gate2_4_pass else "✗"
print(f"\n{status} Gate 2.4 (No duplicates): {'PASS' if gate2_4_pass else 'FAIL'}")


2.4 Duplicate Check:
  Duplicate participant_uuid values: 0

✓ Gate 2.4 (No duplicates): PASS


In [12]:
# Test 2.5: All participants in population table
print("\n2.5 Population Coverage:")
query = "SELECT COUNT(DISTINCT uuid) as total_participants FROM ds.silverdb.participant"
cursor = connection.cursor()
cursor.execute(query)
total_pop = cursor.fetchone()[0]
cursor.close()

print(f"  Participants in output: {len(output_df):,}")
print(f"  Total participants in population table: {total_pop:,}")
print(f"  Coverage: {len(output_df)/total_pop*100:.1f}%")

# Check if all output participants exist in population table
query = f"""
SELECT COUNT(*) as missing_count
FROM ds.polaris.gut_microbiome_diversity d
LEFT JOIN ds.silverdb.participant p ON d.participant_uuid = p.uuid
WHERE p.uuid IS NULL
"""
cursor = connection.cursor()
cursor.execute(query)
missing_from_pop = cursor.fetchone()[0]
cursor.close()

print(f"  Participants NOT in population table: {missing_from_pop}")

gate2_5_pass = missing_from_pop == 0
status = "✓" if gate2_5_pass else "✗"
print(f"\n{status} Gate 2.5 (All in population table): {'PASS' if gate2_5_pass else 'FAIL'}")


2.5 Population Coverage:


  Participants in output: 12,302
  Total participants in population table: 46,302
  Coverage: 26.6%


  Participants NOT in population table: 1

✗ Gate 2.5 (All in population table): FAIL


In [13]:
# Overall Gate 2 result
gate2_pass = all([gate2_1_pass, gate2_2_pass, gate2_3_pass, gate2_4_pass, gate2_5_pass])
print(f"\n✓ Gate 2 (QC Gates): {'PASS' if gate2_pass else 'FAIL'}")


✓ Gate 2 (QC Gates): FAIL


## Gate 3: Citation Verification
#
**Test:** Verify all clinical claims in manifest have valid citations

In [14]:
print("\n" + "=" * 80)
print("CITATION VERIFICATION")
print("=" * 80)

# Read manifest
import pathlib
notebook_dir = pathlib.Path().absolute()
manifest_path = notebook_dir / 'manifest.md'
with open(manifest_path, 'r') as f:
    manifest_content = f.read()

# Extract footnote references
footnote_refs = re.findall(r'\[\^(\d+)\]', manifest_content)
unique_refs = sorted(set(int(ref) for ref in footnote_refs))

# Extract footnote definitions
footnote_defs = re.findall(r'^\s*-\s*\[\^(\d+)\]:\s*(.+)$', manifest_content, re.MULTILINE)
footnote_dict = {int(num): text.strip() for num, text in footnote_defs}

print(f"\nFootnote references found: {len(unique_refs)}")
print(f"Footnote definitions found: {len(footnote_dict)}")

# Check all references have definitions
missing_defs = [ref for ref in unique_refs if ref not in footnote_dict]
if missing_defs:
    print(f"  ✗ Missing definitions for: {missing_defs}")
else:
    print(f"  ✓ All {len(unique_refs)} references have definitions")

# Check all definitions have URLs or DOIs
citations_with_urls = 0
for num, text in footnote_dict.items():
    if 'http' in text or 'doi' in text.lower() or 'DOI' in text:
        citations_with_urls += 1

print(f"\nCitations with URLs/DOIs: {citations_with_urls}/{len(footnote_dict)}")

gate3_pass = (len(missing_defs) == 0) and (citations_with_urls == len(footnote_dict))
print(f"\n✓ Gate 3 (Citation verification): {'PASS' if gate3_pass else 'FAIL'}")


CITATION VERIFICATION

Footnote references found: 10
Footnote definitions found: 10
  ✓ All 10 references have definitions

Citations with URLs/DOIs: 9/10

✓ Gate 3 (Citation verification): FAIL


## Gate 4: Reproducibility
#
**Test:** Re-execute computation and verify results match output table

In [15]:
print("\n" + "=" * 80)
print("REPRODUCIBILITY VALIDATION")
print("=" * 80)

# Sample 1000 participants for reproducibility check
np.random.seed(123)
repro_sample = np.random.choice(output_df['participant_uuid'].values, size=min(1000, len(output_df)), replace=False)

print(f"\nReproducibility sample: {len(repro_sample)} participants")


REPRODUCIBILITY VALIDATION

Reproducibility sample: 1000 participants


In [16]:
# Fetch data and recompute
placeholders = ','.join([f"'{p}'" for p in repro_sample])
query = f"""
WITH most_recent_samples AS (
    SELECT
        participant_uuid,
        MAX(sample_id) as max_sample_id
    FROM ds.omics.gut_mb_metaphlan
    WHERE level = 'species'
        AND mock = false
        AND participant_uuid IN ({placeholders})
    GROUP BY participant_uuid
)
SELECT
    m.participant_uuid,
    m.species,
    m.relative_abundance
FROM ds.omics.gut_mb_metaphlan m
INNER JOIN most_recent_samples mrs
    ON m.participant_uuid = mrs.participant_uuid
    AND m.sample_id = mrs.max_sample_id
WHERE m.level = 'species'
    AND m.mock = false
    AND m.species IS NOT NULL
    AND m.relative_abundance > 0
"""

cursor = connection.cursor()
cursor.execute(query)
repro_raw = pd.DataFrame(cursor.fetchall(), columns=[desc[0] for desc in cursor.description])
cursor.close()

# Recompute diversity
repro_results = []
for participant_uuid, group in repro_raw.groupby('participant_uuid'):
    abundances = group['relative_abundance'].values
    shannon = calculate_shannon(abundances)
    species_count = len(abundances)

    diversity_category = "Low diversity" if shannon < 3.0 else "Normal diversity"
    quality_flag = "Low quality/dysbiotic" if species_count < 20 else "Adequate"

    repro_results.append({
        'participant_uuid': participant_uuid,
        'shannon': shannon,
        'species': species_count,
        'div_cat': diversity_category,
        'qual_flag': quality_flag,
    })

repro_df = pd.DataFrame(repro_results)

In [17]:
# Compare with output table
repro_compare = output_df[output_df['participant_uuid'].isin(repro_sample)].copy()
repro_compare = repro_compare.merge(repro_df, on='participant_uuid', how='inner')

# Check row counts match
print(f"Output table rows: {len(repro_compare)}")
print(f"Recomputed rows: {len(repro_df)}")
rows_match = len(repro_compare) == len(repro_df)

# Check values match
shannon_match = np.allclose(
    repro_compare['gut_microbiome_diversity__shannon_index'].values,
    repro_compare['shannon'].values,
    rtol=0, atol=1e-6
)
species_match = (repro_compare['gut_microbiome_diversity__observed_species'] == repro_compare['species']).all()
div_cat_match = (repro_compare['gut_microbiome_diversity__diversity_category'] == repro_compare['div_cat']).all()
qual_flag_match = (repro_compare['gut_microbiome_diversity__quality_flag'] == repro_compare['qual_flag']).all()

print(f"\nComparison results:")
print(f"  ✓ Row counts match: {rows_match}")
print(f"  ✓ Shannon indices match (within 1e-6): {shannon_match}")
print(f"  ✓ Species counts match: {species_match}")
print(f"  ✓ Diversity categories match: {div_cat_match}")
print(f"  ✓ Quality flags match: {qual_flag_match}")

gate4_pass = all([rows_match, shannon_match, species_match, div_cat_match, qual_flag_match])
print(f"\n✓ Gate 4 (Reproducibility): {'PASS' if gate4_pass else 'FAIL'}")

Output table rows: 1000
Recomputed rows: 1000

Comparison results:
  ✓ Row counts match: True
  ✓ Shannon indices match (within 1e-6): True
  ✓ Species counts match: True
  ✓ Diversity categories match: True
  ✓ Quality flags match: True

✓ Gate 4 (Reproducibility): PASS


## Summary

In [18]:
print("\n" + "=" * 80)
print("VALIDATION SUMMARY")
print("=" * 80)

summary = pd.DataFrame({
    'Gate': [
        '1. Correctness',
        '2. QC Gates',
        '3. Citation Verification',
        '4. Reproducibility',
    ],
    'Status': [
        'PASS' if gate1_pass else 'FAIL',
        'PASS' if gate2_pass else 'FAIL',
        'PASS' if gate3_pass else 'FAIL',
        'PASS' if gate4_pass else 'FAIL',
    ]
})

print(summary.to_string(index=False))

all_pass = all([gate1_pass, gate2_pass, gate3_pass, gate4_pass])
print(f"\n{'✓' if all_pass else '✗'} Overall validation: {'PASS' if all_pass else 'FAIL'}")


VALIDATION SUMMARY
                    Gate Status
          1. Correctness   PASS
             2. QC Gates   FAIL
3. Citation Verification   FAIL
      4. Reproducibility   PASS

✗ Overall validation: FAIL


## Run Configuration

In [19]:
print("\n" + "=" * 80)
print("RUN CONFIGURATION")
print("=" * 80)

print("\nUser prompt:")
print('  "Create a curated phenotype for **gut microbiome diversity** using the `create-curated-phenotype` skill."')

print("\nSkills used:")
print("  - create-curated-phenotype v0.2.0")
print("  - hpp-datasets v1.1")
print("  - databricks (MCP tool)")


RUN CONFIGURATION

User prompt:
  "Create a curated phenotype for **gut microbiome diversity** using the `create-curated-phenotype` skill."

Skills used:
  - create-curated-phenotype v0.2.0
  - hpp-datasets v1.1
  - databricks (MCP tool)


## Process Metrics

In [20]:
print("\n" + "=" * 80)
print("PROCESS METRICS")
print("=" * 80)

# Wall-clock time
start_time = datetime.fromisoformat("2026-02-22T15:54:33+00:00")
end_time = datetime.now(timezone.utc)
elapsed = end_time - start_time

print(f"\nWall-clock time:")
print(f"  Start: {start_time.isoformat()}Z")
print(f"  End: {end_time.isoformat()}Z")
print(f"  Elapsed: {elapsed}")

print(f"\nHuman interventions: 0")
print(f"  (Autonomous execution - no human input required)")

print(f"\nErrors encountered: 0")
print(f"  (Clean execution from fresh start)")


PROCESS METRICS

Wall-clock time:
  Start: 2026-02-22T15:54:33+00:00Z
  End: 2026-02-23T13:34:03.856095+00:00Z
  Elapsed: 21:39:30.856095

Human interventions: 0
  (Autonomous execution - no human input required)

Errors encountered: 0
  (Clean execution from fresh start)


In [21]:
# Close connection
connection.close()
print("\n✓ Validation complete")


✓ Validation complete
